# 00 — Foundations: Attention and Generation (A-Day 1 + A-Day 2)

Pure PyTorch notebook — no vLLM. Build scaled dot-product attention, multi-head attention, and the autoregressive generation loop from scratch.

**Goals:**
- Implement scaled dot-product attention and verify against `F.scaled_dot_product_attention`
- Build a decoder block and understand tensor shapes
- Implement the token generation loop by hand (no `model.generate()`)
- Add greedy, temperature, and top-k sampling
- Measure tokens/second baseline for later comparison with vLLM

## 1. Scaled dot-product attention (A-Day 1)

Implement: $\text{Attention}(Q,K,V) = \text{softmax}(QK^T / \sqrt{d}) V$

In [ ]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention_manual(Q, K, V, mask=None):
    """
    Q, K, V: (batch, seq_len, d_k). For decoder self-attention, seq_len can be 1 (decode) or N (prefill).
    """
    d_k = Q.size(-1)
    scale = math.sqrt(d_k)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / scale  # (B, seq_q, seq_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
    attn_weights = F.softmax(scores, dim=-1)
    out = torch.matmul(attn_weights, V)  # (B, seq_q, d_k)
    return out

# Example: 3 tokens, d=64
B, N, d = 1, 3, 64
Q = torch.randn(B, N, d)
K = torch.randn(B, N, d)
V = torch.randn(B, N, d)
out_manual = scaled_dot_product_attention_manual(Q, K, V)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)
print("Q shape:", Q.shape, "| K shape:", K.shape, "| V shape:", V.shape)
print("Output shape:", out_manual.shape)
print("Match SDPA:", torch.allclose(out_manual, out_sdpa, atol=1e-5))

**Tensor shapes for 3-token input:** `Q,K,V` are each `[1, 3, 64]`. Scores `QK^T` are `[1, 3, 3]`. After softmax and multiply by V, output is `[1, 3, 64]`.

## 2. Multi-head attention (A-Day 1)

Split hidden dim across heads, run attention per head, concatenate and project.

In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_q = torch.nn.Linear(d_model, d_model)
        self.W_k = torch.nn.Linear(d_model, d_model)
        self.W_v = torch.nn.Linear(d_model, d_model)
        self.W_o = torch.nn.Linear(d_model, d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def _reshape_heads(self, x):
        B, S, D = x.shape
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)  # (B, H, S, d_k)

    def _reshape_back(self, x):
        B, H, S, d_k = x.shape
        return x.transpose(1, 2).contiguous().view(B, S, self.d_model)

    def forward(self, x, mask=None):
        B, S, _ = x.shape
        Q = self._reshape_heads(self.W_q(x))
        K = self._reshape_heads(self.W_k(x))
        V = self._reshape_heads(self.W_v(x))
        scale = math.sqrt(self.d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = torch.matmul(attn, V)
        out = self._reshape_back(out)
        return self.W_o(out)

# Demo: 4 heads, d_model=64
mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(1, 5, 64)
y = mha(x)
print("Input:", x.shape, "-> Output:", y.shape)

## 3. Decoder block forward pass (A-Day 1)

One block: self-attention + residual + LayerNorm, then FFN + residual + LayerNorm. Print shapes.

In [ ]:
class DecoderBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.0):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ln1 = torch.nn.LayerNorm(d_model)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_ff),
            torch.nn.GELU(),
            torch.nn.Linear(d_ff, d_model),
        )
        self.ln2 = torch.nn.LayerNorm(d_model)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Self-attention + residual + LayerNorm
        attn_out = self.self_attn(x, mask=mask)
        x = self.ln1(x + self.dropout(attn_out))
        # FFN + residual + LayerNorm
        ffn_out = self.ffn(x)
        x = self.ln2(x + self.dropout(ffn_out))
        return x

d_model, num_heads, d_ff = 64, 4, 256
block = DecoderBlock(d_model, num_heads, d_ff)
x = torch.randn(1, 3, d_model)
print("Input:", x.shape)
z = block(x)
print("Output:", z.shape)

## 4. Autoregressive generation loop (A-Day 2)

Load Qwen2.5-1.5B tokenizer and model. Implement the generate loop manually: repeatedly call `model.forward()`, take the logits for the last position, pick next token, append, repeat until EOS or max_new_tokens. No `model.generate()`.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
model.eval()
print("Model and tokenizer loaded.")

In [ ]:
def generate_manual(
    model,
    tokenizer,
    prompt: str,
    max_new_tokens: int = 50,
    do_sample: bool = False,
    temperature: float = 1.0,
    top_k: int = 50,
) -> str:
    """Autoregressive loop: no past_key_values (no KV cache yet)."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]  # (1, seq_len)
    generated = list(input_ids[0].tolist())
    eos_id = tokenizer.eos_token_id

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=torch.tensor([generated], device=model.device))
            logits = outputs.logits[:, -1, :]  # (1, vocab_size)
            if do_sample:
                if temperature > 0:
                    logits = logits / temperature
                if top_k > 0:
                    v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                    logits[logits < v[:, -1:]] = float('-inf')
                probs = F.softmax(logits, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1).item()
            else:
                next_token = logits.argmax(dim=-1).item()
            generated.append(next_token)
            if next_token == eos_id:
                break
    return tokenizer.decode(generated, skip_special_tokens=True)

In [ ]:
prompt = "The capital of France is"
out_greedy = generate_manual(model, tokenizer, prompt, max_new_tokens=30, do_sample=False)
print("Greedy:", out_greedy)

## 5. Greedy vs sampling (A-Day 2)

Same prompt: greedy (argmax), temperature sampling, top-k sampling.

In [ ]:
print("Greedy:", generate_manual(model, tokenizer, prompt, max_new_tokens=25, do_sample=False))
print("Temperature=0.7:", generate_manual(model, tokenizer, prompt, max_new_tokens=25, do_sample=True, temperature=0.7, top_k=0))
print("Top-k=10:", generate_manual(model, tokenizer, prompt, max_new_tokens=25, do_sample=True, temperature=1.0, top_k=10))

## 6. Tokens/second baseline (A-Day 2)

Time 100 generated tokens at batch=1 (no KV cache). This is the baseline to compare against in later notebooks.

In [ ]:
import time

def measure_tokens_per_second(model, tokenizer, prompt, num_tokens=100, warmup=5):
    # Warmup
    for _ in range(warmup):
        generate_manual(model, tokenizer, prompt, max_new_tokens=2)
    # Timed run
    start = time.perf_counter()
    generate_manual(model, tokenizer, prompt, max_new_tokens=num_tokens)
    elapsed = time.perf_counter() - start
    return num_tokens / elapsed

tps_baseline = measure_tokens_per_second(model, tokenizer, "Hello, world.", num_tokens=100)
print(f"Baseline (no KV cache, batch=1): {tps_baseline:.2f} tokens/sec")